## Silver Layer — Activity Details
Denormalized fact table joining activities with gear and weather data.

**Purpose:** One row per activity with all context for analysis and dashboards.

**Sources:**
* `silver.activities` - base activity metrics
* `silver.gear_list` - cleaned gear inventory  
* `silver.activity_weather` - cleaned weather (Celsius, km/h)

**Join strategy:** Left joins to preserve all activities (not all have gear or weather)

In [0]:
from pyspark.sql.functions import col, when, get_json_object
from pyspark.sql.types import BooleanType

In [0]:
# Read base activities from silver layer
df_activities = spark.table("garmin_lakehouse.silver.activities")

print(f"Base activities: {df_activities.count()} rows")

In [0]:
# Step 1: Join activities with activity_gear bridge table
df_activity_gear = spark.table("garmin_lakehouse.silver.activity_gear").select(
    col("activity_id"),
    col("gear_pk"),
    col("gear_id")
)

df_with_activity_gear = df_activities.join(
    df_activity_gear, 
    df_activities["activity_id"] == df_activity_gear["activity_id"], 
    "left"
).drop(df_activity_gear["activity_id"])  # Drop duplicate activity_id column

# Step 2: Join with gear_list to get gear details
df_gear = spark.table("garmin_lakehouse.silver.gear_list").select(
    col("gear_pk"),
    col("gear_name"),
    col("gear_type"),
    col("gear_status")
)

df_with_gear = df_with_activity_gear.join(df_gear, "gear_pk", "left")

print(f"After gear joins: {df_with_gear.count()} rows")

In [0]:
# Left join with cleaned weather data from silver layer
df_weather = spark.table("garmin_lakehouse.silver.activity_weather").select(
    col("activity_id"),
    col("temperature_c"),
    col("feels_like_c"),
    col("wind_speed_kmh"),
    col("weather_condition")
)

df_with_weather = df_with_gear.join(df_weather, "activity_id", "left")

print(f"After weather join: {df_with_weather.count()} rows")

In [0]:
# Add derived columns
df_enriched = df_with_weather \
    .withColumn("has_gear", col("gear_pk").isNotNull()) \
    .withColumn("has_weather", col("temperature_c").isNotNull())

print(f"Enriched activities: {df_enriched.count()} rows")
print(f"Activities with gear: {df_enriched.filter(col('has_gear') == True).count()}")
print(f"Activities with weather: {df_enriched.filter(col('has_weather') == True).count()}")

In [0]:
# Write to silver.activity_details
df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("garmin_lakehouse.silver.activity_details")

row_count = spark.table("garmin_lakehouse.silver.activity_details").count()
print(f"✅ silver.activity_details written: {row_count} rows")

In [0]:
# Display sample data
df_sample = spark.table("garmin_lakehouse.silver.activity_details")

print(f"Total columns: {len(df_sample.columns)}")
print(f"\nColumn list:")
for col_name in df_sample.columns:
    print(f"  - {col_name}")

print(f"\nSample data (10 most recent activities):")
display(df_sample.orderBy(col("start_date").desc()).limit(10))